<a href="https://colab.research.google.com/github/AlexisReyesUpe/neural-network-image-classifier/blob/main/AE1_CD3_I1_EI_%3CD%3E_%3C_AlexisReyes%3ECopy__Codigo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importar las librerias requeridas
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras import optimizers
from keras.models import Sequential, load_model
from keras.layers import Dense, Dropout, Flatten, Activation
from keras.layers import Convolution2D, MaxPooling2D
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# Definir la ruta de las imagenes
Entrenamiento ="/content/drive/MyDrive/MyEI/Imagenes/Entrenar"
Validacion ="/content/drive/MyDrive/MyEI/Imagenes/Validar"

# Definir los Hiperparametros de la arquitectura CNN
epocas = 100  # Más épocas con early stopping
altura, anchura = 150, 150
batch_size = 16  # Batch size más pequeño
pasos = 100

# Definir los hiperparametros de las capas convolucionales
kernel1 = 32
kernel1_size = (3,3)
kernel2 = 64
kernel2_size = (3,3)
size_pooling = (2,2)

clases = 6  # orgánico, vidrio, metal, papel, plástico, basura

In [ ]:
# Generar datos sinteticos con normalización correcta (1/255 en lugar de 1/200)
entrenar = ImageDataGenerator(
    rescale=1./255,  # Normalización estándar
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,  # Añadido flip vertical
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    brightness_range=[0.8, 1.2],  # Variación de brillo
    fill_mode='nearest'
)

validar = ImageDataGenerator(rescale=1./255)  # Misma normalización para validación

# Leemos las imagenes
imagenes_entrenamiento = entrenar.flow_from_directory(
    Entrenamiento,
    target_size=(altura, anchura),
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True
)

imagenes_validacion = validar.flow_from_directory(
    Validacion,
    target_size=(altura, anchura),
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

# Guardar el mapeo de clases para uso posterior
clases_indices = imagenes_entrenamiento.class_indices
clases_invertido = {v: k for k, v in clases_indices.items()}

print("Clases encontradas:", clases_indices)
print(f"Total imágenes de entrenamiento: {imagenes_entrenamiento.samples}")
print(f"Total imágenes de validación: {imagenes_validacion.samples}")

# Calcular pesos de clase para manejar desbalance
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Obtener etiquetas reales
etiquetas = np.array(imagenes_entrenamiento.classes)
clases_unicas = np.unique(etiquetas)

# Calcular pesos
pesos_clase = compute_class_weight(
    class_weight='balanced',
    classes=clases_unicas,
    y=etiquetas
)

# Convertir a diccionario
pesos_clase_dict = {i: peso for i, peso in zip(clases_unicas, pesos_clase)}
print("Pesos de clase:", pesos_clase_dict)

Found 374 images belonging to 6 classes.
Found 110 images belonging to 6 classes.
Clases encontradas: {'aluminio': 0, 'basura': 1, 'organicos': 2, 'papel': 3, 'plástico': 4, 'vidrio': 5}
Total imágenes de entrenamiento: 374
Total imágenes de validación: 110
Pesos de clase: {np.int32(0): np.float64(0.9739583333333334), np.int32(1): np.float64(0.9894179894179894), np.int32(2): np.float64(1.1333333333333333), np.int32(3): np.float64(0.9739583333333334), np.int32(4): np.float64(0.9739583333333334), np.int32(5): np.float64(0.9739583333333334)}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Construir arquitectura de la CNN - Simplificada para evitar sobreajuste
ModeloCNN = Sequential()

# Primera capa convolucional
ModeloCNN.add(Convolution2D(32, (3, 3), padding="same", input_shape=(altura, anchura, 3), activation="relu"))
ModeloCNN.add(MaxPooling2D(pool_size=(2, 2)))

# Segunda capa convolucional
ModeloCNN.add(Convolution2D(64, (3, 3), padding="same", activation="relu"))
ModeloCNN.add(MaxPooling2D(pool_size=(2, 2)))

# Tercera capa convolucional
ModeloCNN.add(Convolution2D(128, (3, 3), padding="same", activation="relu"))
ModeloCNN.add(MaxPooling2D(pool_size=(2, 2)))

# Aplanar
ModeloCNN.add(Flatten())

# Red neuronal multicapa
ModeloCNN.add(Dense(256, activation="relu"))
ModeloCNN.add(Dropout(0.5))  # Mayor dropout para evitar sobreajuste

ModeloCNN.add(Dense(128, activation="relu"))
ModeloCNN.add(Dropout(0.5))

# Capa de salida
ModeloCNN.add(Dense(clases, activation="softmax"))

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Establecer parametros de entrenamiento
ModeloCNN.compile(
    loss="categorical_crossentropy",
    optimizer=optimizers.Adam(learning_rate=0.0001),  # Learning rate más bajo
    metrics=["accuracy"]
)

# Callbacks para mejorar el entrenamiento
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

callbacks = [early_stopping, reduce_lr]

In [ ]:
# Calcular steps dinámicamente
pasos_entrenamiento = imagenes_entrenamiento.samples // batch_size
pasos_validacion = imagenes_validacion.samples // batch_size

# Entrenar modelo CNN con pesos de clase para manejar desbalance
historial = ModeloCNN.fit(
    imagenes_entrenamiento,
    steps_per_epoch=pasos_entrenamiento,
    epochs=epocas,
    validation_data=imagenes_validacion,
    validation_steps=pasos_validacion,
    callbacks=callbacks,
    class_weight=pesos_clase_dict,  # Usar pesos de clase
    verbose=1
)

# Imprimir solo el resultado final (sin gráficas)
print(f"Precisión final en entrenamiento: {historial.history['accuracy'][-1]:.4f}")
print(f"Precisión final en validación: {historial.history['val_accuracy'][-1]:.4f}")

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 118s 5s/step - accuracy: 0.2040 - loss: 1.8709 - val_accuracy: 0.3125 - val_loss: 1.7517 - learning_rate: 1.0000e-04
Epoch 2/100
 1/23 ━━━━━━━━━━━━━━━━━━━━ 18s 853ms/step - accuracy: 0.1875 - loss: 1.7626

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


23/23 ━━━━━━━━━━━━━━━━━━━━ 5s 210ms/step - accuracy: 0.1875 - loss: 1.7626 - val_accuracy: 0.3125 - val_loss: 1.7532 - learning_rate: 1.0000e-04
Epoch 3/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - accuracy: 0.1926 - loss: 1.8015 - val_accuracy: 0.3854 - val_loss: 1.7597 - learning_rate: 1.0000e-04
Epoch 4/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 116ms/step - accuracy: 0.0625 - loss: 1.8245 - val_accuracy: 0.4167 - val_loss: 1.7597 - learning_rate: 1.0000e-04
Epoch 5/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.2087 - loss: 1.7842 - val_accuracy: 0.3542 - val_loss: 1.7363 - learning_rate: 1.0000e-04
Epoch 6/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 151ms/step - accuracy: 0.1250 - loss: 1.7868 - val_accuracy: 0.4479 - val_loss: 1.7321 - learning_rate: 1.0000e-04
Epoch 7/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 40s 1s/step - accuracy: 0.2563 - loss: 1.7348 - val_accuracy: 0.5625 - val_loss: 1.6927 - learning_rate: 1.0000e-04
Epoch 8/100
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 148ms/step - accuracy: 0.0625 - lo

In [ ]:
# Guardar el modelo entrenado de la CNN
ModeloCNN.save("/content/drive/MyDrive/MyEI/Imagenes/Modelo/modeloCNN8DEI.h5")
ModeloCNN.save_weights("/content/drive/MyDrive/MyEI/Imagenes/Modelo/pesos.weights.h5")

# Guardar el mapeo de clases
import json
with open("/content/drive/MyDrive/MyEI/Imagenes/Modelo/clases_indices.json", "w") as f:
    json.dump(clases_indices, f)

print("Modelo y pesos guardados correctamente")

Modelo y pesos guardados correctamente


In [ ]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
from keras.models import load_model

def evaluar_imagen(ruta_imagen, modelo_path=None, pesos_path=None, modelo=None):
    """Evalúa una imagen y devuelve solo la clase predicha"""
    # Cargar el modelo si no se proporciona
    if modelo is None:
        modelo = load_model(modelo_path)
        if pesos_path:
            modelo.load_weights(pesos_path)

    # Preprocesar la imagen
    altura, anchura = 150, 150
    imagen = load_img(ruta_imagen, target_size=(altura, anchura))
    img_array = img_to_array(imagen)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0  # Normalización correcta

    # Predecir
    prediccion = modelo.predict(img_array, verbose=0)

    # Obtener la clase con mayor probabilidad
    clase_idx = np.argmax(prediccion[0])

    # Mapeo de clases
    clases = {
        0: 'orgánico',
        1: 'vidrio',
        2: 'metal',
        3: 'papel',
        4: 'plástico',
        5: 'basura'
    }

    # Intentar cargar el mapeo guardado
    try:
        with open("/content/drive/MyDrive/MyEI/Imagenes/Modelo/clases_indices.json", "r") as f:
            clases_indices = json.load(f)
            clases_invertido = {int(v): k for k, v in clases_indices.items()}
            clase_nombre = clases_invertido[clase_idx]
    except:
        clase_nombre = clases[clase_idx]

    # Mostrar solo el resultado
    print(f"Reconoció: {clase_nombre}")
    return clase_nombre

# Ejemplo de uso
imagen_path = "/content/drive/MyDrive/MyEI/Imagenes/Validar/organicos/limon14.jpg"
modelo_path = "/content/drive/MyDrive/MyEI/Imagenes/Modelo/modeloCNN8DEI.h5"
pesos_path = "/content/drive/MyDrive/MyEI/Imagenes/Modelo/pesos.weights.h5"

evaluar_imagen(imagen_path, modelo_path, pesos_path)

/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Reconoció: aluminio


'aluminio'

In [ ]:
def evaluar_directorio(directorio, modelo_path, pesos_path=None, num_imagenes=None):
    """Evalúa todas las imágenes en un directorio y muestra solo estadísticas básicas"""
    # Cargar el modelo
    modelo = load_model(modelo_path)
    if pesos_path:
        modelo.load_weights(pesos_path)

    # Obtener lista de imágenes
    imagenes = []
    for root, _, files in os.walk(directorio):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                imagenes.append(os.path.join(root, file))

    # Limitar número de imágenes si se especifica
    if num_imagenes and len(imagenes) > num_imagenes:
        imagenes = imagenes[:num_imagenes]

    # Estadísticas
    total = len(imagenes)
    resultados = {}

    # Procesar cada imagen
    for i, img_path in enumerate(imagenes):
        # Obtener clase real del nombre del directorio
        clase_real = os.path.basename(os.path.dirname(img_path)).lower()

        # Evaluar imagen (sin mostrar resultado individual)
        clase_pred = evaluar_imagen(img_path, modelo=modelo)

        # Registrar resultado
        if clase_real not in resultados:
            resultados[clase_real] = {'total': 0, 'correctas': 0}

        resultados[clase_real]['total'] += 1
        if clase_real == clase_pred.lower():
            resultados[clase_real]['correctas'] += 1

    # Mostrar solo resultados finales
    print("\n--- RESULTADOS DE EVALUACIÓN ---")
    print(f"Total de imágenes evaluadas: {total}")

    total_correctas = sum(r['correctas'] for r in resultados.values())
    precision_global = total_correctas / total if total > 0 else 0

    print(f"Precisión global: {precision_global:.2%}")
    print("\nResultados por clase:")

    for clase, stats in resultados.items():
        precision = stats['correctas'] / stats['total'] if stats['total'] > 0 else 0
        print(f"  {clase}: {precision:.2%} ({stats['correctas']}/{stats['total']})")

# Ejemplo de uso
directorio_validacion = "/content/drive/MyDrive/MyEI/Imagenes/Validar"
modelo_path = "/content/drive/MyDrive/MyEI/Imagenes/Modelo/modeloCNN8DEI.h5"
pesos_path = "/content/drive/MyDrive/MyEI/Imagenes/Modelo/pesos.weights.h5"

evaluar_directorio(directorio_validacion, modelo_path, pesos_path, num_imagenes=50)

Reconoció: papel
Reconoció: papel
Reconoció: papel
Reconoció: vidrio
Reconoció: papel
Reconoció: papel
Reconoció: papel
Reconoció: organicos
Reconoció: aluminio
Reconoció: papel
Reconoció: papel
Reconoció: papel
Reconoció: papel
Reconoció: papel
Reconoció: organicos
Reconoció: plástico
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: vidrio
Reconoció: aluminio
Reconoció: papel
Reconoció: vidrio
Reconoció: aluminio
Reconoció: vidrio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: aluminio
Reconoció: vidrio
Reconoció: aluminio
Reconoció: vidrio
Reconoció: vidrio
Reconoció: aluminio
Reconoció: organicos
Reconoció: organicos

--- RESULTADOS DE EVALUACIÓN ---
Total de im

In [ ]:
!pip install gradio
import gradio as gr
from tensorflow.keras.utils import load_img, img_to_array
import numpy as np
from keras.models import load_model
import json

# Cargar el modelo
modelo_path = "/content/drive/MyDrive/MyEI/Imagenes/Modelo/modeloCNN8DEI.h5"
pesos_path = "/content/drive/MyDrive/MyEI/Imagenes/Modelo/pesos.weights.h5"

modeloCNN = load_model(modelo_path)
modeloCNN.load_weights(pesos_path)

# Intentar cargar el mapeo de clases
try:
    with open("/content/drive/MyDrive/MyEI/Imagenes/Modelo/clases_indices.json", "r") as f:
        clases_indices = json.load(f)
        clases_invertido = {int(v): k for k, v in clases_indices.items()}
except:
    # Mapeo por defecto si no se encuentra el archivo
    clases_invertido = {
        0: 'orgánico',
        1: 'vidrio',
        2: 'metal',
        3: 'papel',
        4: 'plástico',
        5: 'basura'
    }

# Función para predecir imagen
def predecir_imagen(imagen):
    if imagen is None:
        return "Por favor, sube una imagen"

    # Preprocesar imagen
    imagen = imagen.resize((150, 150))
    imagen = img_to_array(imagen) / 255.0  # Normalización correcta
    imagen = np.expand_dims(imagen, axis=0)

    # Predecir
    pred = modeloCNN.predict(imagen, verbose=0)
    clase_idx = np.argmax(pred)
    clase = clases_invertido[clase_idx]

    # Devolver solo el resultado
    return f"Reconoció: {clase}"

# Interfaz Gradio
interfaz = gr.Interface(
    fn=predecir_imagen,
    inputs=gr.Image(type="pil", label="Sube una imagen de reciclaje"),
    outputs=gr.Textbox(label="Resultado"),
    title="Clasificador de Residuos",
    description="Reconoce orgánico, vidrio, metal, papel, plástico y basura."
)

interfaz.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88e33f0bf6bf7b160f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
